# WaterOne Water Quality & River Telemetry Lakehouse Analysis

This notebook demonstrates analytical consumption directly from the **WaterOne Gold Lakehouse Marts** via DuckDB in **3 lines of SQL**.

**PWS ID**: KS2009110 (Water District 1 of Johnson County, Kansas)

In [1]:
import duckdb
import pandas as pd

# Connect to embedded local DuckDB catalog
con = duckdb.connect("../database/water_lakehouse.duckdb", read_only=True)
print("Connected to Lakehouse catalog!")

## 1. Executive Finished Water Quality Compliance
Querying  joined to .

In [2]:
query_compliance = """
SELECT 
    c.display_name,
    s.result_value,
    c.unit,
    c.mcl as epa_mcl,
    c.action_level,
    c.warning_threshold,
    s.met_standard
FROM fact_water_quality_samples s
JOIN dim_contaminant c ON s.contaminant_id = c.contaminant_id
WHERE s.result_value IS NOT NULL
ORDER BY c.category, c.display_name;
"""
df_compliance = con.execute(query_compliance).df()
df_compliance

## 2. Upstream River Stress & Source Blending Metrics
Querying .

In [3]:
query_river = """
SELECT 
    date,
    kansas_river_flow_cfs,
    missouri_river_flow_cfs,
    blended_turbidity_ntu,
    turbidity_7d_ma,
    treatment_stress_ratio,
    water_safety_index
FROM mart_daily_water_quality
ORDER BY date DESC
LIMIT 10;
"""
df_river = con.execute(query_river).df()
df_river

## 3. Pre-Breach ML Anomaly Warnings (Isolation Forest)
Querying .

In [4]:
query_anomalies = """
SELECT 
    date,
    severity_index,
    alert_level,
    primary_contributing_factor,
    recommended_action
FROM mart_anomaly_events
WHERE alert_level != 'NORMAL'
ORDER BY severity_index DESC;
"""
df_anomalies = con.execute(query_anomalies).df()
df_anomalies